In [1]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import webbrowser
import os

# =====================================================================
# 🎛️ MAP CONFIGURATION
# =====================================================================
INPUT_FILE = '/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Vectorized_Mars7_Forecast_2026_2031.csv'  # Your 7-parameter output
MAP_OUTPUT = 'Global_Seismic_Stress_2026_2031.html'
MIN_PROBABILITY_TO_PLOT = 75.0  # Focus on High and Extreme risks
# =====================================================================

def generate_seismic_risk_map():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found. Run the Forecaster script first.")
        return

    # 1. Load the Forecast Data
    df = pd.read_csv(INPUT_FILE)
    df = df[df['Peak_Stress_Probability'] >= MIN_PROBABILITY_TO_PLOT]
    
    # 2. Initialize the Global Map (Centered at 0, 0)
    m = folium.Map(location=[15, 0], zoom_start=2, tiles='cartodbpositron')

    # 3. Add a Heatmap Layer to visualize "Regional Hotspots"
    # This represents the "Proximity" of stress around the coordinates
    heat_data = [[row['Latitude'], row['Longitude'], row['Peak_Stress_Probability']] for index, row in df.iterrows()]
    HeatMap(heat_data, radius=25, blur=15, gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(m)

    # 4. Add Interactive Markers for Top Events
    marker_cluster = MarkerCluster().add_to(m)

    for index, row in df.iterrows():
        # Assign Color based on your Probability Ranges
        color = 'red' if row['Peak_Stress_Probability'] >= 85 else 'orange'
        
        # Create a detailed Popup for the "Black Swan" event
        popup_text = f"""
        <div style="font-family: Arial; width: 200px;">
            <h4 style="margin-bottom:5px;">{row['Threatened_Zone']}</h4>
            <hr style="margin:5px 0;">
            <b>Date:</b> {row['Risk_Date']}<br>
            <b>AI Probability:</b> {row['Peak_Stress_Probability']}%<br>
            <b>Category:</b> {"EXTREME" if color == 'red' else "HIGH"}<br>
            <b>Coordinates:</b> {row['Latitude']}, {row['Longitude']}
        </div>
        """
        
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=8,
            popup=folium.Popup(popup_text, max_width=250),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
        ).add_to(marker_cluster)

    # 5. Save and Launch
    m.save(MAP_OUTPUT)
    print(f"✅ Interactive Map generated: {MAP_OUTPUT}")
    
    # Optional: Automatically open in browser (if local)
    # webbrowser.open('file://' + os.path.realpath(MAP_OUTPUT))

if __name__ == "__main__":
    generate_seismic_risk_map()

✅ Interactive Map generated: Global_Seismic_Stress_2026_2031.html
